## Chapter 3 of the Series: Three Axis Movement Control of a Drone

### Introduction
When we want to move a drone, we must determine the thrust each motor should supply. Moving thorough the z axis, we observed that having the same thrust from all the motors was enough. However, if we wanto to achieve the movement in multiple axis, such as from $ (x_1,y_1,z_1) $ to $ (x_2,y_2,z_2) $ , we must calculate the thrust using the **vectors**. First, we must define a position vector for the center of mass of the object, in this case it is:

$$ p = (x,y,z) $$

$ p* $ is the fixed target position where:

$$ \lim_{t \to \infty} ​p(t)=p∗ $$

A drone, in this case a quadrotor, produces a single scalar thrust with magnitude $ T $ directed along its $ z $ axis plus three torques $ (\tau_x, \tau_y, \tau_z) $ generated by differential rotor speeds. Because it is impossible for a drone to directly apply force to the $ x $ or  $ y $ axis, we must move drone through these axis by tilting awy from the $ z $ axis. 

### Rigid Body Model
We will consider the drone as a rigid body of mass $ m $ and moment of inertia of $ I = (I_x,I_y,I_z) $. The translational equation of motion is:

$$ m(\ddot p) = R(\phi, \theta, \psi)T_B - mg \hat{z} $$

In this formula, $ \hat{z} $ defines the direction of up. Combined with the weight, $ mg $, it shows the downward force applied by the Earth's gravity toward the drone. $ T_B = (0,0,T) $ is the thrust vector. The drone can only push itself on the $ z $ axis, therefore, only $ T_z $ is a non-zero value. Addition of pitch, yaw, and, roll $ (\phi, \theta, \psi) $ enabşes drone to move to the other directions. It is important tonote that $R(\phi, \theta, \psi) \in \text{SO}(3)$ which means that matrix rotates the vector without stretching it. By locking yaw at $ \psi = 0 $, we can make the drone move in only $ x $ and $ y $ axis. According to the Euler Angle Convention:

$$  R = R_z(\psi)\times R_y(\theta)\times R_x(\phi) $$

As mentioned above, setting $ R_z(0) $ results in:

$$ R(\theta) = \begin{bmatrix}
\cos\theta & 0 & \sin\theta \\
0 & 1 & 0 \\
-\sin\theta & 0 & \cos\theta
\end{bmatrix} $$

Rotational Equation for a standard rigid body is:

$$  I\dot{w} + w\times (Iw) = \tau $$

$ w = (p, q, r) $ where $ p $, $ q $, $ r $ represents angular velocity on roll, pitch, and yaw. Likewise, torque also has 3 axis: $ \tau = (\tau _x, \tau _y, \tau _z) $ . For small maneuvers, the gyroscopic coupling term $ w \times (Iw) $ is dropped. Giving:

$$ I_{xx} \dot{p} = \tau _x $$
$$ I_{yy} \dot{q} = \tau _y $$

Last chapter, we derived the equation for movemment in $ z $ axis, damped oscillation:

$$ \ddot{e} + kd\times \dot{e} + kp\times e = 0 $$

### From One Axis to Two
The matrix only used pitch($ \theta $) which is only sufficient for forward-backward motion. However, we also want sidways motion which comes from roll($ \phi $). Multiplying these two will give us small-tilt rotation matrix. 

$$ R(\phi,\theta,0) = R_y(\theta)\times R_x(\phi) = \begin{bmatrix} \cos\theta & \sin\theta\sin\phi & \sin\theta\cos\phi \\ 0 & \cos\phi & -\sin\phi \\ -\sin\theta & \cos\theta\sin\phi & \cos\theta\cos\phi \end{bmatrix} $$

This may look complicating -even intimidating to some extent; however, we only need a single column of the matrix: The thrust vector is $ T_B = R(0, 0, T) $, so multiplying $ R $ by $ T_B $ will just pick out the third column of $ R $ scaled by $ T $:

$$R(\phi, \theta, 0) \times T_B = T \times (\sin\theta \cos\phi, -\sin\phi, \cos\theta \cos\phi)$$

This is the thrust direction in the world frame, for any tilt angle.

### An Assumption For Small Angles
For very small angles, we can assume:

$$ \sin 0 \approx 0 $$
$$ \cos 0 \approx 1 $$

Applying this gives us:

$$R(\phi, \theta, 0) \times T_B \approx T \times (\theta, -\phi, 1)$$

Since the drone is hovering in equilibrium($ T \approx mg $) substituting into the etranslational equation from before gives us:

$$ \ddot{x} = -g\times \theta $$
$$ \ddot{y} = -g\times \phi $$

In other words, pitching the nose down accelerates the drone forward, and rolling to one side accelerates it sideways. Both of these are proportional to the gravity.

**CAUTION**: It is important to note that our equation only holds true if $ x $ depends entirely on $ \theta $, and $ y $ depends entirely on $ \phi $. In other words, we assume that they don't interfere with each other, as long as angles stay negligeble.

### The Outer Loop: How Much Should The Drone Lean
We will use the same idea from the chapter 2, steer a number toward a target poition. Unlike the chapter 2, output is the desired acceleration rather than the thrust.

$$\begin{aligned} \ddot{x}_{\text{des}}^{*} &= k_{p,xy} \, e_x - k_{d,xy} \, \dot{x} \\ \ddot{y}_{\text{des}}^{*} &= k_{p,xy} \, e_y - k_{d,xy} \, \dot{y} \end{aligned}$$

where $ e_x = x^* - x $ and $ e_y = y^* - y $, exactly like the height error from Chapter 2. We can't directly comman an acceleration to the drone. We can only command it how much to tilt. By using the derived equation from small angle assumption, we can convert our desired accelertion into a desired lean angle:

$$\theta_{\text{des}} = \frac{\ddot{x}_{\text{des}}^{*}}{g}, \qquad \phi_{\text{des}} = -\frac{\ddot{y}_{\text{des}}^{*}}{g}$$

In our code, we used `max_tilt` to clmp these into a maximum value. There is two reasons for that: Safety, and accuracy. Tilting so much may cause drone to crash. In addition to that, angles over 15-20 may cause our assumptions to fail.

### The Inner Loop
This loop controls the angular movement. From rotatonal equation of motion:

$$I_{yy} \dot{q} = \tau_y \implies \ddot{\theta} = \frac{\tau_y}{I_{yy}}$$

Applying the exact same idea from above on angle error instead of position error:

$$\ddot{\theta}_{\text{des}} = k_{p,\text{att}} (\theta_{\text{des}} - \theta) - k_{d,\text{att}} \, q$$

This gives us the desired angular acceleration. However, we need thrust instead of acceleration:

$$\tau_y = I_{yy} \ddot{\theta}_{\text{des}}$$

This multipication is significant for the drone. A small drone's $ I_{yy} $ is a tiny number. Thus, skipping this step would make the drone move uncontrollably. Substituting torque bck into the rotational equation of motion. Instead of thrust, same equation now describes angle settles:

$$\ddot{e}_{\theta} + k_{d,\text{att}} \dot{e}_{\theta} + k_{p,\text{att}} e_{\theta} = 0$$

### Reason for The Requirement of 2 Seperate Loops

The tilting happens much faster than moving. Because of the timescale gap, innerloop must be thrustworthy while the outerloop can ensure the movement is hapening.

![Drone Gif](/workspace/isaaclab/learn/Drone_movement_3_axis.gif)
